# 12.1 Saving ML Models

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 12: Model Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-12-model-deployment/12.01_saving-ml-models.ipynb)

Companion code for Section 12.1 of the book. Run the cells in order from top to bottom.

**Example 12.1.1**

In [ ]:
import numpy as np
import pickle
import joblib
import os
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import time

# ── Train a model to save ─────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Build pipeline: scaler + model together (prevents leakage at inference time)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingClassifier(n_estimators=200, random_state=42))
])
pipeline.fit(X_tr, y_tr)
print(f'Model trained. Test accuracy: {pipeline.score(X_te, y_te):.4f}')

# ── Save with Pickle ──────────────────────────────────────────────
with open('model_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)

# Load with Pickle
with open('model_pipeline.pkl', 'rb') as f:
    loaded_pkl = pickle.load(f)
print(f'Pickle loaded. Accuracy: {loaded_pkl.score(X_te, y_te):.4f}')

# ── Save with Joblib (recommended for sklearn) ────────────────────
joblib.dump(pipeline, 'model_pipeline.joblib', compress=3)  # compress 0-9
loaded_jl = joblib.load('model_pipeline.joblib')
print(f'Joblib loaded. Accuracy: {loaded_jl.score(X_te, y_te):.4f}')

# Compare file sizes and load times
pkl_size = os.path.getsize('model_pipeline.pkl') / 1024
jl_size  = os.path.getsize('model_pipeline.joblib') / 1024
print(f'\nFile sizes:')
print(f'  Pickle:  {pkl_size:.1f} KB')
print(f'  Joblib:  {jl_size:.1f} KB  ({(1-jl_size/pkl_size)*100:.1f}% smaller with compress=3)')

# Load time comparison
for name, path, loader in [
    ('Pickle', 'model_pipeline.pkl',    lambda p: pickle.load(open(p,'rb'))),
    ('Joblib', 'model_pipeline.joblib', joblib.load),
]:
    t0 = time.time()
    for _ in range(100): loader(path)
    avg_ms = (time.time()-t0)/100*1000
    print(f'  {name} avg load time: {avg_ms:.2f}ms per load')

**Example 12.1.2**

In [ ]:
import json
import hashlib
import datetime
import joblib
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def save_model_with_metadata(pipeline, X_test, y_test, model_name,
                               feature_names, class_names, save_dir='models'):
    '''
    Save model with full metadata for versioning and reproducibility.
    '''
    import os
    os.makedirs(save_dir, exist_ok=True)

    # Generate version string
    timestamp  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    model_path = f'{save_dir}/{model_name}_v{timestamp}.joblib'

    # Compute model hash for integrity verification
    joblib.dump(pipeline, model_path)
    with open(model_path, 'rb') as f:
        model_hash = hashlib.sha256(f.read()).hexdigest()[:16]

    # Evaluate on test set
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else None

    # Build metadata
    metadata = {
        'model_name':    model_name,
        'version':       timestamp,
        'model_path':    model_path,
        'model_hash':    model_hash,
        'created_at':    datetime.datetime.now().isoformat(),
        'python_version':__import__('sys').version,
        'sklearn_version':__import__('sklearn').__version__,
        'feature_names': list(feature_names),
        'class_names':   list(class_names),
        'n_features':    len(feature_names),
        'n_test_samples':len(y_test),
        'metrics': {
            'accuracy': float(accuracy_score(y_test, y_pred)),
            'auc':      float(roc_auc_score(y_test, y_prob)) if y_prob is not None else None,
        },
        'pipeline_steps': [step[0] for step in pipeline.steps],
    }

    # Save metadata as JSON alongside model
    meta_path = model_path.replace('.joblib', '_metadata.json')
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f'Model saved: {model_path}')
    print(f'Metadata:    {meta_path}')
    print(f'Hash:        {model_hash}')
    print(f'Metrics:     accuracy={metadata["metrics"]["accuracy"]:.4f}  auc={metadata["metrics"]["auc"]:.4f}')

    return model_path, metadata


def load_model_with_verification(model_path, expected_hash=None):
    '''Load model and verify integrity via hash check.'''
    # Load metadata
    meta_path = model_path.replace('.joblib', '_metadata.json')
    with open(meta_path) as f:
        metadata = json.load(f)

    # Verify hash
    with open(model_path, 'rb') as f:
        actual_hash = hashlib.sha256(f.read()).hexdigest()[:16]

    if expected_hash and actual_hash != expected_hash:
        raise ValueError(f'Model integrity check FAILED! Expected {expected_hash}, got {actual_hash}')

    if actual_hash != metadata['model_hash']:
        raise ValueError('Model file may have been tampered with!')

    model = joblib.load(model_path)
    print(f'Model loaded and verified: {model_path}')
    print(f'Version: {metadata["version"]}  Accuracy: {metadata["metrics"]["accuracy"]:.4f}')
    return model, metadata


# Demonstrate
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
pipe = Pipeline([('sc', StandardScaler()), ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))])
pipe.fit(X_tr, y_tr)

path, meta = save_model_with_metadata(
    pipe, X_te, y_te, 'cancer_classifier',
    data.feature_names, data.target_names
)

loaded_model, loaded_meta = load_model_with_verification(path)

**Example 12.1.3**

In [ ]:
# pip install skl2onnx onnxruntime
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnxruntime as rt
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    import time

    data = load_breast_cancer()
    X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_tr.astype(np.float32), y_tr)

    # Convert to ONNX
    initial_type = [('float_input', FloatTensorType([None, X_tr.shape[1]]))]
    onnx_model = convert_sklearn(rf, initial_types=initial_type,
                                  options={id(rf): {'zipmap': False}})

    with open('rf_model.onnx', 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print('ONNX model saved: rf_model.onnx')

    # Load and run with ONNX Runtime
    sess = rt.InferenceSession('rf_model.onnx')
    input_name = sess.get_inputs()[0].name

    X_test_float = X_te.astype(np.float32)

    # Sklearn prediction
    t0 = time.time()
    for _ in range(100): sklearn_pred = rf.predict(X_te)
    sklearn_time = (time.time()-t0)/100*1000

    # ONNX prediction
    t0 = time.time()
    for _ in range(100): onnx_pred = sess.run(None, {input_name: X_test_float})[0]
    onnx_time = (time.time()-t0)/100*1000

    print(f'Sklearn accuracy: {(sklearn_pred == y_te).mean():.4f}')
    print(f'ONNX accuracy:    {(onnx_pred == y_te).mean():.4f}')
    print(f'Predictions match: {(sklearn_pred == onnx_pred).all()}')
    print(f'Sklearn inference: {sklearn_time:.2f}ms/batch')
    print(f'ONNX inference:    {onnx_time:.2f}ms/batch  ({sklearn_time/onnx_time:.1f}x faster)')

except ImportError:
    print('Install: pip install skl2onnx onnxruntime')
    print('ONNX enables: C++, Java, Go, C#, JavaScript inference of Python-trained models')